## Transformación de Datos

In [1]:
#Librerias
import pandas as pd
pd.set_option('display.max_columns', None)  #Mostrar todas las columnas

### Preliminar
- Importación del dataset

In [2]:
df = pd.read_csv("/home/sabri/Escritorio/ADALAB/3.Proyectos/Promo-69-Modulo-3-team-1/EDA/hr.csv")    #Cargar el dataset, adecuar la ruta.
df.head()   #Mostrar las primeras filas del dataset

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,Y,Yes,11,3,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,Y,No,23,4,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,Y,Yes,15,3,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,rESEARCH sCIENTIST,3.0,Married,2909.0,23159,1,Y,Yes,11,3,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,lABORATORY tECHNICIAN,2.0,Married,3468.0,16632,9,Y,No,12,3,4,80.0,1,6,3.0,3,2,2,2,2.0


In [ ]:
# ------------------------------------------------------------
# Eliminar columnas constantes o sin valor
# ------------------------------------------------------------
def drop_columnas(df):
    """
    Elimina columnas que no aportan información:
    """
    # Columnas identificadas en el EDA (ajusta según tu dataset)
    columnas_drop = ['EmployeeCount', 'Over18', 'StandardHours']  # añade más si aparecen
    # También puedes eliminar automáticamente las que tengan un solo valor único
    for col in df.columns:
        if df[col].nunique() == 1:
            columnas_drop.append(col)
    # Eliminar duplicados en la lista y columnas que existan
    columnas_drop = list(set(columnas_drop))
    columnas_drop = [c for c in columnas_drop if c in df.columns]
    return df.drop(columns=columnas_drop)


# ------------------------------------------------------------
# Estandarizar columnas binarias (Yes/No -> 1/0)
# ------------------------------------------------------------
def encode_binary(df):
    """
    Convierte columnas con valores 'Yes'/'No' a 1/0.
    Aplica a 'Attrition' y 'OverTime' (y otras que sean así).
    """
    binary_columnas = ['Attrition', 'OverTime']  # revisa si hay más (ej: 'Gender'? no, ese no es binario)
    mapping = {'Yes': 1, 'No': 0}
    for col in binary_columnas:
        if col in df.columns:
            df[col] = df[col].map(mapping)
    return df


# ------------------------------------------------------------
# Tarea B3: Validar que no queden nulos y que las transformaciones funcionaron
# ------------------------------------------------------------
def validate(df_original, df_clean):
    """
    Verifica:
    - No hay nulos en las columnas clave.
    - La cantidad de filas se mantiene.
    - Las columnas binarias tienen solo 0 y 1.
    """
    # 1. Sin nulos en columnas que deberían estar limpias
    assert df_clean[['Attrition', 'OverTime']].isnull().sum().sum() == 0, "❌ Quedan nulos en Attrition u OverTime"
    # 2. Mismo número de filas
    assert len(df_clean) == len(df_original), "❌ Cambió el número de filas"
    # 3. Columnas binarias solo contienen 0 o 1
    for col in ['Attrition', 'OverTime']:
        if col in df_clean.columns:
            assert set(df_clean[col].unique()).issubset({0,1}), f"❌ {col} tiene valores fuera de {{0,1}}"
    print("✅ Validación superada: todo correcto.")


# ------------------------------------------------------------
# Pipeline completo (integra también funciones de Pareja A)
# ------------------------------------------------------------
def limpiar_dataset(df):
    """
    Aplica todas las transformaciones en orden.
    """
    df_clean = df.copy()
    
    # --- Funciones de Pareja A ---
    # from pareja_A import fix_job_role, fix_types, fix_nulls
    # df_clean = fix_job_role(df_clean)
    # df_clean = fix_types(df_clean)
    # df_clean = fix_nulls(df_clean)
    
    # --- Funciones de Pareja B ---
    df_clean = drop_columnas(df_clean)
    df_clean = encode_binary(df_clean)
    
    # Validación final (necesita el df original para comparar)
    #validate(df, df_clean)     #Habillitar luego de incorporar trtamiento de nulos
    
    return df_clean

# ------------------------------------------------------------
# Ejecución principal (cuando corres el script directamente)
# ------------------------------------------------------------
if __name__ == "__main__":
    # Cargar el CSV original
    df_raw = pd.read_csv("/home/sabri/Escritorio/ADALAB/3.Proyectos/Promo-69-Modulo-3-team-1/EDA/hr.csv")
    print("Shape original:", df_raw.shape)
    
    # Aplicar limpieza completa
    df_limpio = limpiar_dataset(df_raw)
    print("Shape después:", df_limpio.shape)
    print("Primeras filas:\n", df_limpio.head())
    
    # Opcional: guardar resultado
    df_limpio.to_csv("hr_clean.csv", index=False)

Shape original: (1474, 35)
Shape después: (1474, 32)
Primeras filas:
     Age  Attrition     BusinessTravel  DailyRate              Department  \
0  41.0          1      Travel_Rarely       1102                   Sales   
1  49.0          0  Travel_Frequently        279  Research & Development   
2  37.0          1      Travel_Rarely       1373  Research & Development   
3  33.0          0  Travel_Frequently       1392  Research & Development   
4  27.0          0      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeNumber  \
0                 1          2  Life Sciences               1   
1                 8          1  Life Sciences               2   
2                 2          2          Other               4   
3                 3          4  Life Sciences               5   
4                 2          1        Medical               7   

   EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  JobLevel  \
0     

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   str    
 2   BusinessTravel            1357 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1445 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1416 non-null   str    
 8   EmployeeCount             1474 non-null   int64  
 9   EmployeeNumber            1474 non-null   int64  
 10  EnvironmentSatisfaction   1474 non-null   int64  
 11  Gender                    1474 non-null   str    
 12  HourlyRate                1474 non-null   int64  
 13  JobInvolvement            1474 non-null   int64  
 14  JobLevel           

In [9]:
df_limpio

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,1.0,11,3,1,0,8,0.0,1,6,4,0,5.0
1,49.0,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,0.0,23,4,4,1,10,3.0,3,10,7,1,7.0
2,37.0,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,1.0,15,3,2,0,7,3.0,3,0,0,0,0.0
3,33.0,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,rESEARCH sCIENTIST,3.0,Married,2909.0,23159,1,1.0,11,3,3,0,8,3.0,3,8,7,3,0.0
4,27.0,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,lABORATORY tECHNICIAN,2.0,Married,3468.0,16632,9,0.0,12,3,4,1,6,3.0,3,2,2,2,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1469,34.0,0,Travel_Rarely,628,Research & Development,8,3,Medical,2068,2,Male,82,4,2,lABORATORY tECHNICIAN,3.0,Married,4404.0,10228,2,0.0,12,3,1,0,6,3.0,4,4,3,1,2.0
1470,28.0,0,Travel_Rarely,866,Sales,5,3,Medical,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,0.0,18,3,4,0,6,4.0,3,5,4,1,NaN
1471,53.0,0,Travel_Rarely,1084,Research & Development,13,2,Medical,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,0.0,11,3,3,2,5,3.0,3,4,2,1,3.0
1472,24.0,1,Travel_Rarely,240,Human Resources,22,1,Human Resources,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,0.0,11,3,3,1,1,2.0,3,1,0,0,0.0


In [10]:
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1401 non-null   float64
 1   Attrition                 1474 non-null   int64  
 2   BusinessTravel            1357 non-null   str    
 3   DailyRate                 1474 non-null   int64  
 4   Department                1445 non-null   str    
 5   DistanceFromHome          1474 non-null   int64  
 6   Education                 1474 non-null   int64  
 7   EducationField            1416 non-null   str    
 8   EmployeeNumber            1474 non-null   int64  
 9   EnvironmentSatisfaction   1474 non-null   int64  
 10  Gender                    1474 non-null   str    
 11  HourlyRate                1474 non-null   int64  
 12  JobInvolvement            1474 non-null   int64  
 13  JobLevel                  1474 non-null   int64  
 14  JobRole            